### SAC Agent Evaluation on Test Data

In [ ]:
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import random

sys.path.append('..')
from environment.trading_env import TradingEnv
from experts.sac_expert import SACAgent

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
ASSET = "BTCUSDT"  # change to your asset
CONFIG_PATH = "../configs/experts.yaml"
TEST_DATA_PATH = f"../data/processed/{ASSET}/train.csv"
MODEL_PATH = f"../checkpoints/{ASSET}/no_diversity/SAC/sac_expert.pt"

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)
sac_cfg = cfg["sac"]

In [ ]:
# Load test data
test_df = pd.read_csv(TEST_DATA_PATH, index_col=0, parse_dates=True)
print(f"Test data shape: {test_df.shape}")

# Load checkpoint
ckpt = torch.load(MODEL_PATH, map_location='cpu')
print(f"Checkpoint keys: {list(ckpt.keys())}")

# Get expected state dimension from actor network
actor_state = ckpt["actor"]
# Find the first linear layer weight key (actor has 'net' with linear layers)
first_layer_key = None
for key in actor_state.keys():
    if "weight" in key and ("net" in key or "0" in key):
        first_layer_key = key
        break
if first_layer_key is None:
    raise KeyError("Could not find first layer in actor state dict.")

expected_input_dim = actor_state[first_layer_key].shape[1]
expected_features = expected_input_dim - 6   # environment adds 6 state features
print(f"Checkpoint expects {expected_features} features (input dim {expected_input_dim}).")
# Use exact features from config – no automatic adaptation
features = sac_cfg.get("features", [])
if not features:
    raise ValueError("No 'features' defined in config for SAC. Check configs/experts.yaml")

if len(features) != expected_features:
    raise ValueError(
        f"Config features length ({len(features)}) does not match checkpoint "
        f"expected features ({expected_features}).\n"
        "Update your config or use a compatible checkpoint."
    )

# Verify all features exist in the test DataFrame
missing = [f for f in features if f not in test_df.columns]
if missing:
    raise ValueError(f"Missing columns in test data: {missing}")

print("Using features from config:", features)

In [ ]:
asset_type = "crypto" if "USDT" in ASSET else "forex"
window_size = cfg.get("window_size", 20)

env_params = {
    "df": test_df,
    "feature_cols": features,
    "expert_id": 2,                      # SAC = expert 2
    "asset_type": asset_type,
    "contract_size": 1.0 if asset_type == "crypto" else 100000.0,
    "commission_rate": 0.0005 if asset_type == "crypto" else 1.5/100000.0,
    "spread_rate": 0.0002 if asset_type == "crypto" else 0.00005,
    "slippage_rate": 0.0001 if asset_type == "crypto" else 0.00001,
    "leverage": 5 if asset_type == "crypto" else 30,
    "maintenance_margin_ratio": 0.5,
    "initial_balance": 10000.0,
    "window_size": window_size,
    "reward_scale": 1.0,
    "clip_reward": (-10.0, 10.0),
    "max_position_fraction": 0.95,
    "continuous_actions": True,         # SAC uses continuous actions
}
env = TradingEnv(**env_params)

# Instantiate SAC agent with hyperparameters from config
state_dim = len(features) + 6
agent = SACAgent(
    state_dim=state_dim,
    action_dim=1,
    lr=sac_cfg.get("learning_rate", 3e-4),
    gamma=sac_cfg.get("gamma", 0.99),
    tau=sac_cfg.get("tau", 0.005),
    alpha=sac_cfg.get("alpha", 0.2),
    buffer_size=sac_cfg.get("buffer_size", 100000),
    batch_size=sac_cfg.get("batch_size", 256),
    warmup_steps=0,
    max_grad_norm=sac_cfg.get("max_grad_norm", 1.0),
    hidden_dim=sac_cfg.get("hidden_dim", 256),
    num_layers=sac_cfg.get("num_layers", 2),
    activation=sac_cfg.get("activation", "relu"),
    log_std_min=sac_cfg.get("log_std_min", -5),
    log_std_max=sac_cfg.get("log_std_max", 2),
    log_dir=None,
    device="cpu",
)

# Load weights
agent.actor.load_state_dict(ckpt["actor"])
agent.critic.load_state_dict(ckpt["critic"])
agent.critic_target.load_state_dict(ckpt["critic_target"])
if "log_alpha" in ckpt:
    agent.log_alpha.data = ckpt["log_alpha"]
if "total_steps" in ckpt:
    agent._total_steps = ckpt["total_steps"]

print("SAC agent loaded.")

In [ ]:
obs, _ = env.reset()
done = False
equity_curve = [env.initial_balance]   # track equity, not cumulative returns

while not done:
    action = agent.act(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    equity_curve.append(info.get('equity', env.initial_balance))
    done = terminated or truncated

equity_curve = np.array(equity_curve)
cum_return = (equity_curve - equity_curve[0]) / equity_curve[0]   # for plotting

print(f"Final equity: {equity_curve[-1]:.2f}")
print(f"Final cumulative return: {cum_return[-1]:.4f} ({cum_return[-1]*100:.2f}%)")

In [ ]:
# Daily returns (percentage change of equity)
daily_returns = np.diff(equity_curve) / equity_curve[:-1]
total_ret = cum_return[-1]
n = len(daily_returns)
ann_ret = (1 + total_ret) ** (252 / n) - 1 if n > 0 else 0.0
ann_vol = np.std(daily_returns) * np.sqrt(252)
sharpe = ann_ret / ann_vol if ann_vol != 0 else 0.0

# Drawdown (based on equity, not returns)
peak = np.maximum.accumulate(equity_curve)
drawdown = (peak - equity_curve) / peak
max_dd = np.max(drawdown)

win_rate = np.mean(daily_returns > 0)

print("\n--- SAC Performance ---")
print(f"Total Return:       {total_ret:.2%}")
print(f"Annualized Return:  {ann_ret:.2%}")
print(f"Annualized Vol:     {ann_vol:.2%}")
print(f"Sharpe Ratio:       {sharpe:.2f}")
print(f"Max Drawdown:       {max_dd:.2%}")
print(f"Win Rate:           {win_rate:.2%}")

In [ ]:
# Equity curve (cumulative return)
plt.figure(figsize=(14,6))
plt.plot(cum_return, label='SAC', linewidth=2)
plt.title("SAC Equity Curve (Cumulative Return)")
plt.xlabel("Step")
plt.ylabel("Cumulative Return")
plt.grid(True)
plt.legend()
plt.show()

# Drawdown
plt.figure(figsize=(14,3))
plt.fill_between(range(len(drawdown)), 0, drawdown, color='red', alpha=0.4)
plt.plot(drawdown, color='darkred', linewidth=0.5)
plt.title("Drawdown")
plt.xlabel("Step")
plt.ylabel("Drawdown")
plt.grid(True)
plt.show()